In [1]:
from pathlib import Path

def find_lab_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(8):  # достатньо глибини
        if (cur / "data").exists() and (cur / "docs").exists() and (cur / "notebooks").exists():
            return cur
        cur = cur.parent
    raise RuntimeError("Не знайшла lab root (папку з data/docs/notebooks). Запусти notebook з папки project_lab1 або відкрий workspace корінь правильно.")

LAB_ROOT = find_lab_root(Path.cwd())
DATA_DIR = LAB_ROOT / "data"
DOCS_DIR = LAB_ROOT / "docs"

DATA_DIR.mkdir(exist_ok=True)
DOCS_DIR.mkdir(exist_ok=True)

print("CWD:", Path.cwd())
print("LAB_ROOT:", LAB_ROOT)
print("DATA_DIR:", DATA_DIR)

CWD: C:\Users\maia1\data\politiekh\masters\nlp\project_lab1
LAB_ROOT: C:\Users\maia1\data\politiekh\masters\nlp\project_lab1
DATA_DIR: C:\Users\maia1\data\politiekh\masters\nlp\project_lab1\data


In [2]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("KSE-RESEARCH-Group/UAReviews")  # має бути train/test/(може інші)
print(ds)

# зливаємо доступні спліти, які існують
splits = [s for s in ["train", "test", "validation", "challenge"] if s in ds]
df_all = pd.concat([ds[s].to_pandas() for s in splits], ignore_index=True)

print("Rows total:", len(df_all))
print("Columns:", df_all.columns.tolist())

df_all.head(10)

DatasetDict({
    train: Dataset({
        features: ['id', 'rating', 'content', 'source', 'final_emotion', 'final_category', 'length', 'split'],
        num_rows: 11580
    })
})
Rows total: 11580
Columns: ['id', 'rating', 'content', 'source', 'final_emotion', 'final_category', 'length', 'split']


,id,rating,content,source,final_emotion,final_category,length,split
0,622,1.0,Жахливе місце. Жахлива черга. Ніхто нічого під...,original,Anger,Complaint / Dissatisfaction,61,train
1,15442,1.0,"Жахливе відношення до людей,які записані на о...",original,Anger,Complaint / Dissatisfaction,174,train
2,3201,3.0,"Як були черги величезні та багатогодинні, так ...",original,Anger,Complaint / Dissatisfaction,238,train
3,13484,NaN,Сьогодні відвідала ЦНАП(була екскурсія від іні...,original,Happiness,Gratitude / Positive Feedback,241,train
4,6777,1.0,Пиляток на всьому. Ряд об'єктів почали і не до...,original,Anger,Complaint / Dissatisfaction,53,train
5,9905,2.0,"Вступив на ІСТ цього року, тепер молюся, щоб п...",original,Happiness,Question / Request for Help,100,train
6,8422,5.0,Напередодні зареєструвалась в онлайн черзі. Ме...,original,Happiness,Gratitude / Positive Feedback,236,train
7,10001862,NaN,"а по факту ми і дальше просимо боєприпаси,ппо,...",original,Sadness,Question / Request for Help,70,train
8,16607,5.0,"Півгодини перечитував і ""лайкав коментарі"")). ...",original,Happiness,Gratitude / Positive Feedback,231,train
9,9148,5.0,"Команда професійних, ефективних та високомотив...",original,Happiness,Gratitude / Positive Feedback,107,train


In [3]:
# Часто в UAReviews текст може бути в "content" або "text"
# Лейбл може бути "final_category" (категорія) або інше
candidates_text = [c for c in ["content", "text", "review", "comment"] if c in df_all.columns]
candidates_label = [c for c in ["final_category", "category", "label", "final_label"] if c in df_all.columns]
candidates_source = [c for c in ["source", "domain", "platform"] if c in df_all.columns]
candidates_id = [c for c in ["id", "text_id", "uid"] if c in df_all.columns]

print("text candidates:", candidates_text)
print("label candidates:", candidates_label)
print("source candidates:", candidates_source)
print("id candidates:", candidates_id)

text candidates: ['content']
label candidates: ['final_category']
source candidates: ['source']
id candidates: ['id']


In [4]:
TEXT_COL = candidates_text[0]
LABEL_COL = candidates_label[0]
ID_COL = candidates_id[0] if candidates_id else None
SOURCE_COL = candidates_source[0] if candidates_source else None

df = df_all.copy()

if ID_COL is None:
    df["text_id"] = range(1, len(df)+1)
else:
    df = df.rename(columns={ID_COL: "text_id"})

df = df.rename(columns={TEXT_COL: "text", LABEL_COL: "label_raw"})

keep = ["text_id", "text", "label_raw"]
if SOURCE_COL:
    df = df.rename(columns={SOURCE_COL: "source"})
    keep.append("source")

df = df[keep]

df["label_raw"].value_counts().head(20)

label_raw
Gratitude / Positive Feedback    7440
Complaint / Dissatisfaction      2730
Question / Request for Help       615
Neutral Comment                   418
Suggestion / Idea                 377
Name: count, dtype: int64

In [5]:
TEXT_COL = candidates_text[0]
LABEL_COL = candidates_label[0]
ID_COL = candidates_id[0] if candidates_id else None
SOURCE_COL = candidates_source[0] if candidates_source else None

df = df_all.copy()

if ID_COL is None:
    df["text_id"] = range(1, len(df)+1)
else:
    df = df.rename(columns={ID_COL: "text_id"})

df = df.rename(columns={TEXT_COL: "text", LABEL_COL: "label_raw"})

keep = ["text_id", "text", "label_raw"]
if SOURCE_COL:
    df = df.rename(columns={SOURCE_COL: "source"})
    keep.append("source")

df = df[keep]

df["label_raw"].value_counts().head(20)

label_raw
Gratitude / Positive Feedback    7440
Complaint / Dissatisfaction      2730
Question / Request for Help       615
Neutral Comment                   418
Suggestion / Idea                 377
Name: count, dtype: int64

In [6]:
TOP_K = 5
top_labels = df["label_raw"].value_counts().head(TOP_K).index.tolist()
df_top = df[df["label_raw"].isin(top_labels)].copy()

print("Kept labels:", top_labels)
print("Rows after filter:", len(df_top))
df_top["label_raw"].value_counts()

Kept labels: ['Gratitude / Positive Feedback', 'Complaint / Dissatisfaction', 'Question / Request for Help', 'Neutral Comment', 'Suggestion / Idea']
Rows after filter: 11580


label_raw
Gratitude / Positive Feedback    7440
Complaint / Dissatisfaction      2730
Question / Request for Help       615
Neutral Comment                   418
Suggestion / Idea                 377
Name: count, dtype: int64

In [7]:
TARGET_PER_CLASS = 200   # 5 класів -> 1000
RANDOM_STATE = 42

parts = []
for lab, g in df_top.groupby("label_raw"):
    n = min(TARGET_PER_CLASS, len(g))
    parts.append(g.sample(n=n, random_state=RANDOM_STATE))

df_raw = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print("Final size:", len(df_raw))
print(df_raw["label_raw"].value_counts())
df_raw.head(10)

Final size: 1000
label_raw
Neutral Comment                  200
Question / Request for Help      200
Suggestion / Idea                200
Complaint / Dissatisfaction      200
Gratitude / Positive Feedback    200
Name: count, dtype: int64


,text_id,text,label_raw,source
0,922,Зупинили за несправність лівої фари ближнє сві...,Neutral Comment,original
1,10005443,@OscDomesticated @14druzivBanderi ЄВідновлення...,Question / Request for Help,original
2,10003180,А коли можна буде для Сумської області онлайн ...,Question / Request for Help,original
3,10005316,Якщо овдп що в ісу продати раніше терміну випл...,Question / Request for Help,original
4,6169,Академія є єдиним на Україні вищим навчальним ...,Neutral Comment,original
5,10003418,"Сергію, уточніть, будь ласка. ваше запитання д...",Question / Request for Help,original
6,10002246,Боже мій Всемогутній! Ти оберігаєш невинних та...,Question / Request for Help,cosmus
7,3726,Нам було треба зробити оцінку будинку для судо...,Neutral Comment,original
8,12979,Рекомендую юриста Олега Вікторовича по земельн...,Suggestion / Idea,original
9,8433,"Можливо є брак працівників,оскільки прийомом д...",Complaint / Dissatisfaction,original


In [8]:
TARGET_PER_CLASS = 200   # 5 класів -> 1000
RANDOM_STATE = 42

parts = []
for lab, g in df_top.groupby("label_raw"):
    n = min(TARGET_PER_CLASS, len(g))
    parts.append(g.sample(n=n, random_state=RANDOM_STATE))

df_raw = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print("Final size:", len(df_raw))
print(df_raw["label_raw"].value_counts())
df_raw.head(10)

Final size: 1000
label_raw
Neutral Comment                  200
Question / Request for Help      200
Suggestion / Idea                200
Complaint / Dissatisfaction      200
Gratitude / Positive Feedback    200
Name: count, dtype: int64


,text_id,text,label_raw,source
0,922,Зупинили за несправність лівої фари ближнє сві...,Neutral Comment,original
1,10005443,@OscDomesticated @14druzivBanderi ЄВідновлення...,Question / Request for Help,original
2,10003180,А коли можна буде для Сумської області онлайн ...,Question / Request for Help,original
3,10005316,Якщо овдп що в ісу продати раніше терміну випл...,Question / Request for Help,original
4,6169,Академія є єдиним на Україні вищим навчальним ...,Neutral Comment,original
5,10003418,"Сергію, уточніть, будь ласка. ваше запитання д...",Question / Request for Help,original
6,10002246,Боже мій Всемогутній! Ти оберігаєш невинних та...,Question / Request for Help,cosmus
7,3726,Нам було треба зробити оцінку будинку для судо...,Neutral Comment,original
8,12979,Рекомендую юриста Олега Вікторовича по земельн...,Suggestion / Idea,original
9,8433,"Можливо є брак працівників,оскільки прийомом д...",Complaint / Dissatisfaction,original


In [9]:
raw_path = DATA_DIR / "raw.jsonl"
df_raw.to_json(raw_path, orient="records", lines=True, force_ascii=False)
print("Saved:", raw_path)

Saved:

 C:\Users\maia1\data\politiekh\masters\nlp\project_lab1\data\raw.jsonl


In [10]:
# NORMALIZATION

import re

RE_URL = re.compile(r"(https?://\S+|www\.\S+)", re.IGNORECASE)
RE_EMAIL = re.compile(r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b", re.IGNORECASE)
RE_PHONE = re.compile(r"(\+?\d[\d\s\-\(\)]{7,}\d)")

APOSTROPHES = {"’": "'", "ʼ": "'", "`": "'", "´": "'"}

def normalize_text(s: str) -> str:
    s = "" if s is None else str(s)

    for a, b in APOSTROPHES.items():
        s = s.replace(a, b)

    s = RE_URL.sub("<URL>", s)
    s = RE_EMAIL.sub("<EMAIL>", s)
    s = RE_PHONE.sub("<PHONE>", s)

    s = re.sub(r"\s+", " ", s).strip()
    return s

df_proc = df_raw.copy()
df_proc["text_norm"] = df_proc["text"].apply(normalize_text)

df_proc[["text", "text_norm"]].head(10)


,text,text_norm
0,Зупинили за несправність лівої фари ближнє сві...,Зупинили за несправність лівої фари ближнє сві...
1,@OscDomesticated @14druzivBanderi ЄВідновлення...,@OscDomesticated @14druzivBanderi ЄВідновлення...
2,А коли можна буде для Сумської області онлайн ...,А коли можна буде для Сумської області онлайн ...
3,Якщо овдп що в ісу продати раніше терміну випл...,Якщо овдп що в ісу продати раніше терміну випл...
4,Академія є єдиним на Україні вищим навчальним ...,Академія є єдиним на Україні вищим навчальним ...
5,"Сергію, уточніть, будь ласка. ваше запитання д...","Сергію, уточніть, будь ласка. ваше запитання д..."
6,Боже мій Всемогутній! Ти оберігаєш невинних та...,Боже мій Всемогутній! Ти оберігаєш невинних та...
7,Нам було треба зробити оцінку будинку для судо...,Нам було треба зробити оцінку будинку для судо...
8,Рекомендую юриста Олега Вікторовича по земельн...,Рекомендую юриста Олега Вікторовича по земельн...
9,"Можливо є брак працівників,оскільки прийомом д...","Можливо є брак працівників,оскільки прийомом д..."


In [11]:
df_processed = df_proc[["text_id", "text_norm"]].rename(columns={"text_norm": "text"}).copy()

if "source" in df_proc.columns:
    df_processed["source"] = df_proc["source"]

processed_path = DATA_DIR / "processed.csv"
df_processed.to_csv(processed_path, index=False, encoding="utf-8")
print("Saved:", processed_path)

Saved: C:\Users\maia1\data\politiekh\masters\nlp\project_lab1\data\processed.csv


In [12]:
df_labels = df_proc[["text_id", "label_raw"]].rename(columns={"label_raw": "label"}).copy()

labels_path = DATA_DIR / "labels.csv"
df_labels.to_csv(labels_path, index=False, encoding="utf-8")
print("Saved:", labels_path)

df_labels["label"].value_counts()

Saved: C:\Users\maia1\data\politiekh\masters\nlp\project_lab1\data\labels.csv


label
Neutral Comment                  200
Question / Request for Help      200
Suggestion / Idea                200
Complaint / Dissatisfaction      200
Gratitude / Positive Feedback    200
Name: count, dtype: int64

In [13]:
df_audit = df_processed.copy()

df_audit["char_len"] = df_audit["text"].str.len()
df_audit["word_len"] = df_audit["text"].apply(lambda x: len(str(x).split()))

print("N texts:", len(df_audit))
print("\nLength stats (chars):")
print(df_audit["char_len"].describe())

print("\nLength stats (words):")
print(df_audit["word_len"].describe())

print("\nClass distribution:")
print(df_labels["label"].value_counts())

N texts: 1000

Length stats (chars):
count    1000.000000
mean      151.803000
std       114.497082
min         5.000000
25%        82.000000
50%       124.000000
75%       205.000000
max      1678.000000
Name: char_len, dtype: float64

Length stats (words):
count    1000.000000
mean       22.503000
std        18.379548
min         1.000000
25%        12.000000
50%        19.000000
75%        30.000000
max       283.000000
Name: word_len, dtype: float64

Class distribution:
label
Neutral Comment                  200
Question / Request for Help      200
Suggestion / Idea                200
Complaint / Dissatisfaction      200
Gratitude / Positive Feedback    200
Name: count, dtype: int64


In [14]:
dup_mask = df_audit.duplicated(subset=["text"], keep=False)
dup_pct = dup_mask.mean() * 100
print(f"Exact duplicates: {dup_pct:.2f}%")

Exact duplicates: 0.40%


In [15]:
short_mask = df_audit["word_len"] < 5
short_pct = short_mask.mean() * 100
print(f"Too short (<5 words): {short_pct:.2f}%")

Too short (<5 words): 0.20%


In [16]:
RE_ONLY_DIGITS = re.compile(r"^\d+$")
RE_ONLY_SYMBOLS = re.compile(r"^[\W_]+$", re.UNICODE)

def is_garbage(s: str) -> bool:
    s = str(s).strip()
    if s == "":
        return True
    if RE_ONLY_DIGITS.match(s):
        return True
    if RE_ONLY_SYMBOLS.match(s):
        return True
    return False

garbage_mask = df_audit["text"].apply(is_garbage)
garbage_pct = garbage_mask.mean() * 100
print(f"Garbage rows: {garbage_pct:.2f}%")

Garbage rows: 0.00%


In [17]:
df_audit.sample(10, random_state=42)

,text_id,text,source,char_len,word_len
521,365,Дуже чисте та екологічне місце. Де можна добре...,original,57,9
737,6773,"Все б нічого, але як завжди є АЛЕ!! На дворі 2...",original,139,24
740,10002108,Життя дуже часто ставить нас в умови вибору......,original,214,33
660,2659,"По телефону відповіла хамка, підвищує голос, п...",original,83,11
411,10005832,"Куди звернутися одеситам, в яких пошкоджені до...",cosmus,408,62
678,8646,"На оформлення паспорту зробіть ще одне крісло,...",original,118,20
626,15000,"Сподіваюся вартість яку беруть за вхід, витрат...",original,135,21
513,237,"""Гугл"", чомусь вирішив, що відвідування податк...",original,69,8
859,10003704,"Для бізнесу, який надає послуги відповідного К...",original,162,22
136,752,"Співробітник Галина, яка на дверях біля столу ...",original,108,15


In [18]:
for lab in df_labels["label"].unique():
    print("\n", lab, "")
    display(df_proc[df_proc["label_raw"] == lab][["text"]].head(5))


 Neutral Comment 


,text
0,Зупинили за несправність лівої фари ближнє сві...
4,Академія є єдиним на Україні вищим навчальним ...
7,Нам було треба зробити оцінку будинку для судо...
20,Красноградський районий суд Харківської област...
23,"Щоб там щодня, крім вихідних. Нічого особливог..."



 Question / Request for Help 


,text
1,@OscDomesticated @14druzivBanderi ЄВідновлення...
2,А коли можна буде для Сумської області онлайн ...
3,Якщо овдп що в ісу продати раніше терміну випл...
5,"Сергію, уточніть, будь ласка. ваше запитання д..."
6,Боже мій Всемогутній! Ти оберігаєш невинних та...



 Suggestion / Idea 


,text
8,Рекомендую юриста Олега Вікторовича по земельн...
10,"Якщо ви хочете побачити Львів з вершини, підні..."
13,Було б зручно щоб коли у вайбері відміняєш тал...
14,"Краще ""повинна бути зручнішою, ніж Uber чи Boo..."
15,Рекомендую туристам відвідувати. Піднятися на ...



 Complaint / Dissatisfaction 


,text
9,"Можливо є брак працівників,оскільки прийомом д..."
11,Тут так гарно\nНестерпно\nПоруч із тобою бути ...
21,"Тут не про обслуговування ,до того не дійшло,..."
26,"Персонал відверто кажучи хамавитий, некомпетен..."
31,Чому офіційний сайт державного органу України ...



 Gratitude / Positive Feedback 


,text
16,"Рекомендую компанію ТОВ ""БЮРО ОЦІНКИ"", тому що..."
19,Дуже досвідчений та професійний спеціаліст. Ду...
22,Нещодавно взяли пухнастика в Центрі поводження...
24,"Звернулись до Валеріі Олексіївни,дякую за конс..."
25,Були у Романа Володимировича по сімейному пита...


In [19]:
# raw "as in source" for selected ids (1000 rows)
selected_ids = set(df_raw["text_id"].tolist())

source_id_col = "id" if "id" in df_all.columns else "text_id"
raw_source = df_all[df_all[source_id_col].isin(selected_ids)].copy()

if source_id_col != "text_id":
    raw_source = raw_source.rename(columns={source_id_col: "text_id"})

# контроль: має бути рівно 1000
print("raw_source rows:", len(raw_source))
print(raw_source.columns.tolist())

raw_source_path = DATA_DIR / "raw.jsonl"
raw_source.to_json(raw_source_path, orient="records", lines=True, force_ascii=False)
print("Overwritten raw.jsonl as source-like:", raw_source_path)


raw_source rows: 1000
['text_id', 'rating', 'content', 'source', 'final_emotion', 'final_category', 'length', 'split']
Overwritten raw.jsonl as source-like: C:\Users\maia1\data\politiekh\masters\nlp\project_lab1\data\raw.jsonl
